# YOLOv8-OBB Training: Bottle vs Can Detection

**Maqsad:** Roboflow dan olingan oriented bounding box dataseti yordamida butilka va banka aniqlovchi YOLOv8 modelni Google Colab T4 GPU da o'qitish.

**Dataset:** 419 ta labeled rasm (292 train / 61 val / 63 test), 2 klass (bottle, can)

**Model:** YOLOv8n-OBB (nano, ~3M parametrli) — kichik dataset uchun ideal, transfer learning bilan COCO pretrained weights dan boshlaymiz.

**Training plan:** 80 epoch, imgsz=640, batch=16, AdamW, augmentation faol.

## 1. Muhitni tayyorlash

In [ ]:
# GPU borligini tekshirish
!nvidia-smi

In [ ]:
# Ultralytics o'rnatamiz (YOLOv8 va YOLOv8-OBB shu paketda)
%pip install -q ultralytics==8.2.0
import ultralytics
ultralytics.checks()

## 2. Dataset ni Colab ga yuklash

**Variant A — Drive dan:** Loyiha papkasini Google Drive ga yuklang, `data/processed/` papkasini `/content/dataset/` ga ko'chiring.

**Variant B — to'g'ridan-to'g'ri yuklash:** Quyidagi katakni ishga tushirsangiz, lokal kompyuteringizdan zip yuklash dialogi ochiladi.

In [ ]:
# === Variant A: Google Drive dan ulash (TAVSIYA) ===
from google.colab import drive
drive.mount('/content/drive')

# Drive dagi loyiha yo'lini moslashtiring (qaerga yuklaganingizga qarab)
DRIVE_PROJECT = '/content/drive/MyDrive/yolo-portfolio'  # <-- shu yo'lni o'zgartiring agar boshqacha bo'lsa

import shutil, os
if os.path.exists('/content/dataset'):
    shutil.rmtree('/content/dataset')
shutil.copytree(f'{DRIVE_PROJECT}/data/processed', '/content/dataset')
print('Train images:', len(os.listdir('/content/dataset/train/images')))
print('Val images:  ', len(os.listdir('/content/dataset/val/images')))
print('Test images: ', len(os.listdir('/content/dataset/test/images')))

In [ ]:
# data.yaml ni Colab ichida yangidan yaratamiz (absolute path bilan)
yaml_content = '''path: /content/dataset
train: train/images
val: val/images
test: test/images

names:
  0: bottle
  1: can

nc: 2
'''
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)
print(yaml_content)

## 3. Sanity check: bir nechta sample rasmni vizualizatsiya qilish

In [ ]:
import cv2, glob, random
import numpy as np
import matplotlib.pyplot as plt

CLASS_NAMES = {0: 'bottle', 1: 'can'}
COLORS = {0: (0, 200, 0), 1: (0, 100, 255)}

def draw_obb(img, label_path):
    h, w = img.shape[:2]
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 9: continue
            cls = int(parts[0])
            coords = list(map(float, parts[1:9]))
            pts = np.array([[coords[i]*w, coords[i+1]*h] for i in range(0,8,2)], dtype=np.int32)
            cv2.polylines(img, [pts], True, COLORS[cls], 3)
            cv2.putText(img, CLASS_NAMES[cls], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.7, COLORS[cls], 2)
    return img

random.seed(0)
imgs = random.sample(glob.glob('/content/dataset/train/images/*.jpg'), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, imgs):
    lbl = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img = draw_obb(img, lbl)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Training

**Tanlovlar va sabablar:**
- `model='yolov8n-obb.pt'` — nano variant, kichik dataset uchun overfit xavfi kam
- `epochs=80` — 100 dan kam, lekin EarlyStopping (`patience=20`) bilan optimum topiladi
- `imgsz=640` — standart, sifat va tezlik balansi
- `batch=16` — T4 GPU da xotiraga sig'adi
- `optimizer='AdamW'` — kichik datasetlar uchun SGD dan ko'ra barqaror
- `cos_lr=True` — cosine learning rate scheduler, oxirida fine convergence
- `mosaic=1.0, mixup=0.15` — augmentation kuchaytirilgan, kichik dataset uchun muhim

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-obb.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    project='/content/runs',
    name='bottle_can_obb_v1',
    exist_ok=True,
    plots=True,
)

## 5. Test setda baholash

In [ ]:
best = '/content/runs/bottle_can_obb_v1/weights/best.pt'
model = YOLO(best)
metrics = model.val(data='/content/dataset/data.yaml', split='test', plots=True)
print('mAP@0.5      :', metrics.box.map50)
print('mAP@0.5:0.95 :', metrics.box.map)
print('Per-class mAP@0.5:', dict(zip(['bottle','can'], metrics.box.maps)))

## 6. Trained weights va training natijalarini Drive ga saqlash

In [ ]:
import shutil
OUT_DIR = f'{DRIVE_PROJECT}/models'
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(best, f'{OUT_DIR}/best.pt')
# Hammasini ko'chirib qo'yamiz (grafiklar, confusion matrix va h.k.)
if os.path.exists(f'{DRIVE_PROJECT}/results/training'):
    shutil.rmtree(f'{DRIVE_PROJECT}/results/training')
shutil.copytree('/content/runs/bottle_can_obb_v1', f'{DRIVE_PROJECT}/results/training')
print('Saqlandi:', OUT_DIR)

## 7. Sample predictions (README uchun)

In [ ]:
import os
os.makedirs('/content/preds', exist_ok=True)
results = model.predict(
    source='/content/dataset/test/images',
    save=True,
    conf=0.25,
    project='/content/preds',
    name='test_predictions',
    exist_ok=True,
)
shutil.copytree('/content/preds/test_predictions',
                f'{DRIVE_PROJECT}/results/sample_predictions/test',
                dirs_exist_ok=True)
print('Predictions saqlandi.')